# Linear Regression Forecast Model — Departures (Sprint 6)

**Overview**
This notebook implements a Linear Regression model to predict hourly bike departures for downtown stations.

It builds upon the baseline lag-based approach by combining multiple temporal features into a supervised learning model.

**Scope**
- Target variable: **departures**
- Geographic scope: **downtown stations only**
- Temporal granularity: **station-hour level**
- Dataset: precomputed feature table (`downtown_dep_features_v1`)

**Purpose**
The objective is to evaluate whether a simple linear model can outperform the baseline lag predictors.

This model serves as the first machine learning benchmark before more complex models such as Random Forest, XGBoost, and LightGBM.

**Model Approach**

A Linear Regression model is trained using multiple temporal and historical features, including lag-based predictors.

Unlike the baseline, which uses a single lag as prediction, this model learns a weighted combination of features to improve predictive accuracy.

**Feature Set**
The model uses a combination of temporal and historical features, including:

- Lag features (e.g., lag1, lag24, lag168)
- Time-based features (hour, day of week)
- Other engineered predictors available in the dataset

These features allow the model to capture both short-term dynamics and periodic patterns.

In [0]:
# ============================================================
# Linear Regression Model — Departures Forecast
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/downtown_dep_features_v1"
EVAL_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/downtown_dep_linear_regression_progressive_v2"

TRAIN_LOOKBACK_DAYS = 90
TARGET = "target_departures"

# ------------------------------------------------------------
# 1) Load feature dataset
# ------------------------------------------------------------
df_feat = spark.read.parquet(FEATURES_DIR)

loaded_rows = df_feat.count()
print("Loaded feature rows:", f"{loaded_rows:,}")

# ------------------------------------------------------------
# 2) Final feature set
# ------------------------------------------------------------
model_features = [
    "month",
    "hour",
    "dow_num",
    "is_weekend",
    "lag1_dep",
    "lag2_dep",
    "lag3_dep",
    "lag24_dep",
    "lag48_dep",
    "lag168_dep",
    "roll_mean_3h",
    "roll_mean_24h",
    "roll_std_24h"
]

# ------------------------------------------------------------
# 3) Available months
# ------------------------------------------------------------
months_rows = (
    df_feat.select("year", "month")
    .distinct()
    .orderBy("year", "month")
    .collect()
)

months_list = [(int(r["year"]), int(r["month"])) for r in months_rows]

# ------------------------------------------------------------
# 4) Month cache
# ------------------------------------------------------------
month_pd_cache = {}

def load_month_pd(y: int, m: int) -> pd.DataFrame:
    key = (y, m)
    if key in month_pd_cache:
        return month_pd_cache[key]

    sdf = (
        df_feat
        .filter((F.col("year") == y) & (F.col("month") == m))
        .select(
            "station_id", "year", "month", "day", "hour",
            "dow_num", "is_weekend",
            "lag1_dep", "lag2_dep", "lag3_dep",
            "lag24_dep", "lag48_dep", "lag168_dep",
            "roll_mean_3h", "roll_mean_24h", "roll_std_24h",
            TARGET
        )
    )

    pdf = sdf.toPandas()

    if len(pdf) > 0:
        pdf["date"] = pd.to_datetime(pdf[["year", "month", "day"]])

    month_pd_cache[key] = pdf
    return pdf

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end = (test_start - pd.Timedelta(days=1)).to_period("M")
    periods = pd.period_range(start, end, freq="M")
    return [(int(p.year), int(p.month)) for p in periods]

# ------------------------------------------------------------
# 5) Progressive evaluation loop
# ------------------------------------------------------------
results = []

for (y, m) in months_list:
    test_start = pd.Timestamp(year=y, month=m, day=1)
    test_end = test_start + pd.offsets.MonthEnd(0)

    train_end = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month_pd(y, m)
    if test_pd.empty:
        continue

    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        part = load_month_pd(yy, mm)
        if not part.empty:
            train_parts.append(part)

    if not train_parts:
        continue

    train_pd_all = pd.concat(train_parts, ignore_index=True)

    train_pd = train_pd_all[(train_pd_all["date"] >= train_start) & (train_pd_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]

    if train_pd.empty or test_pd_f.empty:
        continue

    # Baseline (lag1) for comparison
    y_true = test_pd_f[TARGET].astype(np.float32).values
    baseline_pred = test_pd_f["lag1_dep"].astype(np.float32).values
    baseline_mae = float(mean_absolute_error(y_true, baseline_pred))
    baseline_rmse = float(np.sqrt(mean_squared_error(y_true, baseline_pred)))

    # Model data
    X_train = train_pd[model_features].astype(np.float32).values
    y_train = train_pd[TARGET].astype(np.float32).values

    X_test = test_pd_f[model_features].astype(np.float32).values
    y_test = test_pd_f[TARGET].astype(np.float32).values

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict
    pred = model.predict(X_test)

    # Metrics
    mae = float(mean_absolute_error(y_test, pred))
    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    improvement_pct = (baseline_mae - mae) / baseline_mae * 100 if baseline_mae else np.nan

    results.append({
        "year": y,
        "month": m,
        "rows_test": int(len(y_test)),
        "baseline_mae": baseline_mae,
        "model_mae": mae,
        "baseline_rmse": baseline_rmse,
        "model_rmse": rmse,
        "improvement_pct": float(improvement_pct)
    })

# ------------------------------------------------------------
# 6) Results
# ------------------------------------------------------------
results_pd = pd.DataFrame(results).sort_values(["year", "month"])
display(results_pd)

# ------------------------------------------------------------
# 7) Global summary
# ------------------------------------------------------------
summary = pd.DataFrame({
    "Metric": [
        "Avg MAE Baseline",
        "Avg MAE Linear Regression",
        "Avg RMSE Baseline",
        "Avg RMSE Linear Regression",
        "Avg Improvement (%)"
    ],
    "Value": [
        results_pd["baseline_mae"].mean(),
        results_pd["model_mae"].mean(),
        results_pd["baseline_rmse"].mean(),
        results_pd["model_rmse"].mean(),
        results_pd["improvement_pct"].mean()
    ]
})

display(summary)

# ------------------------------------------------------------
# 8) Save outputs
# ------------------------------------------------------------
spark.createDataFrame(results_pd).write.mode("overwrite").parquet(EVAL_DIR)
print("Saved Linear Regression evaluation to:", EVAL_DIR)

**Results and Interpretation**

The Linear Regression model is evaluated using MAE and RMSE and compared against the baseline lag-based predictors.

Results indicate whether combining multiple features improves prediction accuracy over naive approaches.

This step establishes a stronger benchmark for subsequent models such as Random Forest and Gradient Boosting methods.